<a href="https://colab.research.google.com/github/madhu-sudhana07/madhu-sudhana07/blob/main/PL_match_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#pip install optuna

In [ ]:
import pandas as pd
import numpy as np
import requests
import io
import warnings
import xgboost as xgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss
import ipywidgets as widgets
from IPython.display import display, clear_output
import datetime

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING) # Hides messy Optuna logs

# ==========================================
# STEP 1: DATA INGESTION (MULTIPLE SEASONS)
# ==========================================
print("1. Fetching historical Premier League data (2020 - Present)...")
seasons = ["2021", "2122", "2223", "2324", "2425", "2526", "2627"]
base_url = "https://www.football-data.co.uk/mmz4281/{}/E0.csv"
dataframes = []

for season in seasons:
    try:
        response = requests.get(base_url.format(season))
        if response.status_code == 200:
            df_season = pd.read_csv(io.StringIO(response.text))
            df_season['Season_Code'] = season
            dataframes.append(df_season)
    except Exception:
        continue

df_all = pd.concat(dataframes, ignore_index=True)
string_cols = df_all.select_dtypes(include=['object']).columns
df_all[string_cols] = df_all[string_cols].apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df_all['Date'] = pd.to_datetime(df_all['Date'], format='mixed', dayfirst=True)
df_all = df_all.sort_values(by=['Date']).reset_index(drop=True)
df_all['Target'] = df_all['FTR'].map({'A': 0, 'D': 1, 'H': 2})

# Keep essential columns
cols_to_keep = ['Season_Code', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Target', 'B365H', 'B365D', 'B365A']
df_master = df_all[cols_to_keep].dropna(subset=['Target']).copy()

# ==========================================
# STEP 2: FEATURE ENGINEERING
# ==========================================
print("2. Engineering multi-season rolling form and rest days...")
home_df = df_master[['Date', 'Season_Code', 'HomeTeam', 'FTHG', 'FTAG']].copy()
home_df.columns = ['Date', 'Season_Code', 'Team', 'Goals_Scored', 'Goals_Conceded']
home_df['Is_Home'] = 1

away_df = df_master[['Date', 'Season_Code', 'AwayTeam', 'FTAG', 'FTHG']].copy()
away_df.columns = ['Date', 'Season_Code', 'Team', 'Goals_Scored', 'Goals_Conceded']
away_df['Is_Home'] = 0

team_timeline = pd.concat([home_df, away_df]).sort_values(by=['Team', 'Date']).reset_index(drop=True)
team_timeline['Rest_Days'] = team_timeline.groupby('Team')['Date'].diff().dt.days.fillna(14)

for col in ['Goals_Scored', 'Goals_Conceded']:
    team_timeline[f'{col}_Rolling_5'] = team_timeline.groupby(['Team', 'Season_Code'])[col].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )

home_features = team_timeline[team_timeline['Is_Home'] == 1].drop(columns=['Is_Home', 'Goals_Scored', 'Goals_Conceded'])
home_features = home_features.rename(columns={'Team': 'HomeTeam'})
home_features.columns = ['Date', 'Season_Code', 'HomeTeam'] + [f'Home_{c}' for c in home_features.columns if c not in ['Date', 'Season_Code', 'HomeTeam']]

away_features = team_timeline[team_timeline['Is_Home'] == 0].drop(columns=['Is_Home', 'Goals_Scored', 'Goals_Conceded'])
away_features = away_features.rename(columns={'Team': 'AwayTeam'})
away_features.columns = ['Date', 'Season_Code', 'AwayTeam'] + [f'Away_{c}' for c in away_features.columns if c not in ['Date', 'Season_Code', 'AwayTeam']]

df_model = df_master.merge(home_features, on=['Date', 'Season_Code', 'HomeTeam'], how='left')
df_model = df_model.merge(away_features, on=['Date', 'Season_Code', 'AwayTeam'], how='left')
df_model = df_model.dropna().reset_index(drop=True)

# ==========================================
# STEP 3: OPTUNA TUNING & MODEL TRAINING
# ==========================================
print("3. Tuning hyperparameters with Optuna and training final XGBoost model (this takes ~30 seconds)...")
features = [
    'Home_Rest_Days', 'Away_Rest_Days',
    'Home_Goals_Scored_Rolling_5', 'Home_Goals_Conceded_Rolling_5',
    'Away_Goals_Scored_Rolling_5', 'Away_Goals_Conceded_Rolling_5'
]

X = df_model[features]
y = df_model['Target'].astype(int)

def objective(trial):
    param = {
        'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': 'mlogloss',
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'random_state': 42
    }
    tscv = TimeSeriesSplit(n_splits=3)
    fold_losses = []
    for train_idx, test_idx in tscv.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        model = xgb.XGBClassifier(**param)
        model.fit(X_tr, y_tr, verbose=False)
        fold_losses.append(log_loss(y_te, model.predict_proba(X_te)))
    return np.mean(fold_losses)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30) # 10 trials for speed. Increase to 30 later for better accuracy.

best_params = study.best_params
best_params.update({'objective': 'multi:softprob', 'num_class': 3, 'random_state': 42})
clf = xgb.XGBClassifier(**best_params)
clf.fit(X, y)
print("✅ Pipeline Complete! Launching UI...\n")

# ==========================================
# STEP 4: INTERACTIVE UI & INFERENCE
# ==========================================
all_teams = sorted(list(set(df_master['HomeTeam'].unique().tolist() + df_master['AwayTeam'].unique().tolist())))

home_dropdown = widgets.Dropdown(options=all_teams, description='Home Team:')
away_dropdown = widgets.Dropdown(options=all_teams, description='Away Team:')
date_picker = widgets.DatePicker(description='Match Date:', value=datetime.date.today() + datetime.timedelta(days=1))
predict_button = widgets.Button(description='Predict Match!', button_style='success', icon='futbol-o')
output_area = widgets.Output()

def get_latest_stats(team_name, is_home, match_date):
    team_matches = df_master[(df_master['HomeTeam'] == team_name) | (df_master['AwayTeam'] == team_name)].sort_values('Date').tail(5)
    rest_days = (match_date - team_matches['Date'].iloc[-1]).days

    scored, conceded = [], []
    for _, row in team_matches.iterrows():
        if row['HomeTeam'] == team_name:
            scored.append(row['FTHG']); conceded.append(row['FTAG'])
        else:
            scored.append(row['FTAG']); conceded.append(row['FTHG'])

    prefix = 'Home_' if is_home else 'Away_'
    return {
        f'{prefix}Rest_Days': float(rest_days),
        f'{prefix}Goals_Scored_Rolling_5': np.mean(scored),
        f'{prefix}Goals_Conceded_Rolling_5': np.mean(conceded),
    }

def on_button_clicked(b):
    with output_area:
        clear_output()
        home = home_dropdown.value
        away = away_dropdown.value
        match_date = pd.to_datetime(str(date_picker.value))

        if home == away:
            print("Error: Home and Away teams cannot be the same!")
            return

        try:
            home_stats = get_latest_stats(home, True, match_date)
            away_stats = get_latest_stats(away, False, match_date)

            input_df = pd.DataFrame([{**home_stats, **away_stats}])[features]
            probs = clf.predict_proba(input_df)[0]

            print(f"--- MATCH PREDICTION: {home} vs {away} ---")
            print(f"1. Home Win ({home}): {probs[2]*100:.2f}%")
            print(f"2. Draw:                 {probs[1]*100:.2f}%")
            print(f"3. Away Win ({away}): {probs[0]*100:.2f}%\n")
        except Exception as e:
            print(f"Could not generate prediction: {e}")

predict_button.on_click(on_button_clicked)
display(widgets.VBox([widgets.HTML(value="<h3>🏆 Premier League Predictor</h3>"), home_dropdown, away_dropdown, date_picker, predict_button, output_area]))

1. Fetching historical Premier League data (2020 - Present)...
2. Engineering multi-season rolling form and rest days...
3. Tuning hyperparameters with Optuna and training final XGBoost model (this takes ~30 seconds)...
✅ Pipeline Complete! Launching UI...



In [ ]:
# Run this in Colab instead of using joblib
# 1. Save the model natively as a JSON
clf.save_model('xgboost_pl_model.json')

# 2. Save the historical data
df_master.to_csv('master_historical_data.csv', index=False)

print("Model saved natively as JSON. Download this file.")

Model saved natively as JSON. Download this file.
